#Silver Layer

In [0]:
%sql


-- 2. Convertir los valores
UPDATE workspace.default.walmart_dataset
SET date_num = to_date(Date, 'dd-MM-yyyy');

-- 3. Solo si el UPDATE terminó correctamente, eliminar la original
ALTER TABLE workspace.default.walmart_dataset
DROP COLUMN Date;

-- 4. Recuperar el nombre original
ALTER TABLE workspace.default.walmart_dataset
RENAME COLUMN date_num TO Date;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_walmart_sales
USING DELTA
AS
SELECT
    t.Store,
    to_date(old.Date, 'dd-MM-yyyy') AS Date,
    t.Weekly_Sales,
    t.Holiday_Flag,
    t.Temperature,
    t.Fuel_Price,
    t.CPI,
    t.Unemployment,
    YEAR(to_date(old.Date, 'dd-MM-yyyy')) AS Year,
    MONTH(to_date(old.Date, 'dd-MM-yyyy')) AS Month
FROM workspace.default.walmart_dataset AS t
-- Recover original Date strings from version 36 (before try_cast destroyed them)
JOIN workspace.default.walmart_dataset@v36 AS old
ON t.record_id = old.record_id
WHERE t.Weekly_Sales IS NOT NULL
    AND t.Weekly_Sales > 0;

In [0]:
%sql
SELECT * FROM silver_walmart_sales LIMIT 10